# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [ ]:
# 상위 폴더 paht 설정 -> common_config.py 모듈 인식을 위해 설정
import sys
sys.path.append("..")
from common_config import llm_connect

model = "gpt-5.4-mini"

temperature = 0
max_tokens = 2086

llm = llm_connect(model=model)

In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
LangGraph는 **LLM 애플리케이션을 그래프 기반으로 설계·실행하는 프레임워크**입니다.  
특히 **에이전트(agent)**, **멀티스텝 워크플로우**, **상태(state)를 가진 실행 흐름**을 만들 때 자주 사용됩니다.

간단히 말하면:

- 일반적인 체인(chain)은 `A → B → C`처럼 **직선형 흐름**에 강하고
- LangGraph는 **분기, 반복, 조건 판단, 상태 업데이트**가 있는 **복잡한 흐름**에 더 적합합니다.

---

## 1. LangGraph가 무엇인가요?

LangGraph는 보통 **LangChain 생태계**에서 많이 쓰이며,  
LLM 호출, 도구(tool) 사용, 메모리 관리, 사람 개입(human-in-the-loop) 등을  
**그래프 구조**로 표현할 수 있게 해줍니다.

즉, 애플리케이션을 다음처럼 만들 수 있습니다:

- 사용자의 질문을 받는다
- 현재 상태를 확인한다
- LLM이 답변할지, 검색 도구를 쓸지 결정한다
- 필요하면 검색을 반복한다
- 중간 결과를 상태에 저장한다
- 최종 답변을 생성한다

이런 흐름을 **노드(node)**와 **엣지(edge)**로 연결해 정의합니다.

---

## 2. 왜 LangGraph를 쓰나요?

LangGraph는 다음 상황에서 유용합니다.

### 2.1 반복과 루프
에이전트는 보통 한 번에 끝나지 않습니다.

예:
- 질문 분석
- 검색
- 검색 결과 평가
- 부족하면 다시 검색
- 충분하면 답변 생성

이런 **루프 구조**를 쉽게 만들 수 있습니다.

### 2.2 상태 기반 실행
중간 결과를 계속 저장하면서 다음 단계 판단에 활용할 수 있습니다.

예:
- 현재까지 수집한 문서
- 이전 도구 호출 결과
- 사용자 승인 여부
- 에러 상태

### 2.3 조건 분기
실행 중 판단에 따라 경로를 바꿀 수 있습니다.

예:
- 단순 질문이면 바로 답변
- 복잡 질문이면 검색 후 답변
- 민감 작업이면 사람 승인 요청

##

In [ ]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있는 도구입니다. LangGraph는 State, Node, Edge 개념을 사용하여 복잡한 AI 파이프라인을 구현합니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```